# Lie vs Coordinate Kinematic Control
## A Fair Comparison on a Planar 4R Arm, with a 3D Euler-Singularity Stress Test

This notebook compares two kinematic controllers under the **same** gains, damping, timestep, and targets:

- **Coordinate controller**: work directly with task coordinates such as $(x, y, \, \phi)$ or ZYX Euler angles.
- **Lie-theoretic controller**: work on $SE(2)$ or $SO(3)$ and define error with the logarithm map.

The goal is not to claim that coordinate methods are always bad. In planar $SE(2)$, a wrapped angle error is already fairly competent. The point is to show:

1. why Lie methods give a cleaner geometric error on rigid-motion groups,
2. why that difference is modest but still visible in the planar 4R case, and
3. why the advantage becomes much more obvious in 3D orientation control near Euler singularities.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'figure.dpi': 150,
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
    'axes.spines.right': False,
    'axes.spines.top': False,
})

SEED = 42
rng = np.random.default_rng(SEED)


In [ ]:
# ========================== Global Constants ==========================

LINK_LENGTHS = np.array([1.0, 0.8, 0.6, 0.4])
N_JOINTS = len(LINK_LENGTHS)

# Planar controller settings
DT_PLANAR = 0.05
PLANAR_GAIN = 4.0
PLANAR_DAMPING = 0.05
PLANAR_MAX_QDOT = 8.0
PLANAR_STEPS = 220

# SO(3) controller settings
DT_SO3 = 0.03
SO3_GAIN = 4.0
SO3_MAX_OMEGA = 3.0
SO3_STEPS = 250

FD_EPSILON = 1e-6

# Shared comparison scenarios
Q0_PLANAR = np.array([0.5, -0.9, 0.3, 0.6])
TARGET_EASY = np.array([1.6, 0.9, 0.8])
TARGET_LARGE = np.array([-0.15637242, -1.52945991, -1.55350010])

Q_FRAME = np.array([0.9, -0.4, 0.1, 0.7])
TARGET_FRAME = np.array([1.2, 1.0, 0.4])
FRAME_ROTATION = np.deg2rad(50.0)

EULER0_SO3 = np.deg2rad([60.0, 89.0, -80.0])
EULERD_SO3 = np.deg2rad([-30.0, 10.0, 100.0])


---
## 1. Planar 4R Setup: Coordinate Rates vs Lie-Group Twists

For the planar 4R arm, the end-effector pose is described by

$$s(q) = [x(q),\ y(q),\ \phi(q)]^\top.$$

The classical differential kinematics uses the **coordinate Jacobian**

$$\dot{s} = J_{\mathrm{coord}}(q)\,\dot{q}.$$

The Lie-group formulation uses the homogeneous transform $T(q) \in SE(2)$ and the **spatial twist Jacobian**

$$V_s = J_{\mathrm{space}}(q)\,\dot{q}, \qquad V_s = \mathrm{vee}(\dot{T}T^{-1}),$$

together with the **body Jacobian**

$$J_{\mathrm{body}}(q) = \mathrm{Ad}_{T(q)^{-1}}\,J_{\mathrm{space}}(q).$$

The distinction matters:

- $J_{\mathrm{coord}}$ maps into raw chart derivatives $(\dot{x}, \dot{y}, \dot{\phi})$.
- $J_{\mathrm{space}}$ maps into a twist on the group itself.
- The Lie controller will use a **body-frame logarithmic pose error**.


In [ ]:
# ========================== Planar Geometry + SE(2) ==========================

J2 = np.array([[0.0, -1.0], [1.0, 0.0]])

def wrap_to_pi(angle):
    """Wrap a scalar or array of angles to [-pi, pi)."""
    return (angle + np.pi) % (2.0 * np.pi) - np.pi


def rot2(theta):
    """2D rotation matrix."""
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])


def fk_planar_coords(q, L):
    """Planar 4R forward kinematics in task coordinates [x, y, phi]."""
    phi = np.cumsum(q)
    x = np.sum(L * np.cos(phi))
    y = np.sum(L * np.sin(phi))
    return np.array([x, y, wrap_to_pi(np.sum(q))])


def all_joint_positions(q, L):
    """Base + every joint + end-effector positions in the plane."""
    phi = np.cumsum(q)
    x = np.concatenate([[0.0], np.cumsum(L * np.cos(phi))])
    y = np.concatenate([[0.0], np.cumsum(L * np.sin(phi))])
    return np.column_stack([x, y])


def jacobian_planar_coords(q, L):
    """Coordinate Jacobian for [x, y, phi]."""
    n = len(q)
    phi = np.cumsum(q)
    J = np.zeros((3, n))
    for k in range(n):
        J[0, k] = -np.sum(L[k:] * np.sin(phi[k:]))
        J[1, k] = np.sum(L[k:] * np.cos(phi[k:]))
        J[2, k] = 1.0
    return J


def pose_se2_from_xyphi(pose):
    """Homogeneous transform in SE(2) from [x, y, phi]."""
    T = np.eye(3)
    T[:2, :2] = rot2(pose[2])
    T[:2, 2] = pose[:2]
    return T


def pose_se2_from_q(q, L):
    """Planar 4R pose represented as T(q) in SE(2)."""
    return pose_se2_from_xyphi(fk_planar_coords(q, L))


def pose_coords_from_se2(T):
    """Recover [x, y, phi] from a planar homogeneous transform."""
    return np.array([T[0, 2], T[1, 2], np.arctan2(T[1, 0], T[0, 0])])


def inv_se2(T):
    """Inverse of a planar homogeneous transform."""
    R = T[:2, :2]
    p = T[:2, 2]
    T_inv = np.eye(3)
    T_inv[:2, :2] = R.T
    T_inv[:2, 2] = -R.T @ p
    return T_inv


def hat_se2(xi):
    """Hat map for xi = [v_x, v_y, omega]."""
    vx, vy, omega = xi
    return np.array([
        [0.0, -omega, vx],
        [omega, 0.0, vy],
        [0.0, 0.0, 0.0],
    ])


def vee_se2(X):
    """Vee map inverse to hat_se2."""
    return np.array([X[0, 2], X[1, 2], X[1, 0]])


def exp_se2(xi):
    """Closed-form exponential map exp: se(2) -> SE(2)."""
    v = np.asarray(xi[:2])
    omega = float(xi[2])
    T = np.eye(3)
    T[:2, :2] = rot2(omega)
    if abs(omega) < 1e-10:
        V = np.eye(2)
    else:
        A = np.sin(omega) / omega
        B = (1.0 - np.cos(omega)) / omega
        V = A * np.eye(2) + B * J2
    T[:2, 2] = V @ v
    return T


def log_se2(T):
    """Closed-form logarithm map log: SE(2) -> se(2)."""
    R = T[:2, :2]
    p = T[:2, 2]
    omega = np.arctan2(R[1, 0], R[0, 0])
    if abs(omega) < 1e-10:
        v = p
    else:
        A = np.sin(omega) / omega
        B = (1.0 - np.cos(omega)) / omega
        denom = A * A + B * B
        V_inv = (A * np.eye(2) - B * J2) / denom
        v = V_inv @ p
    return hat_se2(np.array([v[0], v[1], omega]))


def adjoint_se2(T):
    """Adjoint action Ad_T on planar twists [v_x, v_y, omega]."""
    R = T[:2, :2]
    p = T[:2, 2]
    Ad = np.eye(3)
    Ad[:2, :2] = R
    Ad[:2, 2] = np.array([p[1], -p[0]])
    return Ad


def jacobian_planar_space(q, L):
    """Spatial twist Jacobian: vee(Tdot T^{-1}) = J_space(q) qdot."""
    pose = fk_planar_coords(q, L)
    J_coord = jacobian_planar_coords(q, L)
    J_space = J_coord.copy()
    J_space[0, :] += pose[1] * J_coord[2, :]
    J_space[1, :] += -pose[0] * J_coord[2, :]
    return J_space


def jacobian_planar_body(q, L):
    """Body twist Jacobian: vee(T^{-1} Tdot) = J_body(q) qdot."""
    T = pose_se2_from_q(q, L)
    return adjoint_se2(inv_se2(T)) @ jacobian_planar_space(q, L)


def damped_pinv(J, damping):
    """Damped least-squares pseudoinverse."""
    m = J.shape[0]
    return J.T @ np.linalg.inv(J @ J.T + damping**2 * np.eye(m))


def ee_path_length(pose_hist):
    """Arc length of the end-effector path in the plane."""
    return np.sum(np.linalg.norm(np.diff(pose_hist[:, :2], axis=0), axis=1))


---
## 2. Planar Controllers

The **coordinate controller** uses the chart error

$$e_{\mathrm{coord}} = [x_d - x,\ y_d - y,\ \mathrm{wrap}(\phi_d - \phi)]^\top,$$

and solves

$$\dot{q} = J_{\mathrm{coord}}(q)^\#\,K\,e_{\mathrm{coord}}.$$

The **Lie controller** uses the body-frame relative pose

$$T_e = T(q)^{-1}T_d, \qquad e_{\mathrm{Lie}} = \mathrm{vee}(\log(T_e)),$$

and solves

$$\dot{q} = J_{\mathrm{body}}(q)^\#\,K\,e_{\mathrm{Lie}}.$$

Both use the same damping, gain, timestep, and velocity limits. Only the representation of task error changes.


In [ ]:
# ========================== Planar Controllers + Simulation ==========================

def coordinate_pose_error(current_pose, target_pose):
    """Task-coordinate error [dx, dy, dphi] with wrapped angle."""
    error = target_pose - current_pose
    error[2] = wrap_to_pi(error[2])
    return error


def lie_pose_error_body(T_current, T_target):
    """Body-frame SE(2) logarithmic pose error."""
    return vee_se2(log_se2(inv_se2(T_current) @ T_target))


def simulate_planar_coordinate_control(
    q0,
    target_pose,
    L,
    dt=DT_PLANAR,
    gain=PLANAR_GAIN,
    damping=PLANAR_DAMPING,
    steps=PLANAR_STEPS,
    max_qdot=PLANAR_MAX_QDOT,
):
    """Roll out the coordinate-space controller on the planar 4R arm."""
    q = np.asarray(q0, dtype=float).copy()

    q_hist = [q.copy()]
    pose_hist = [fk_planar_coords(q, L)]
    error_hist, error_norm = [], []
    qdot_hist, qdot_norm = [], []
    converged_step = None

    for k in range(steps):
        pose = fk_planar_coords(q, L)
        error = coordinate_pose_error(pose, target_pose)
        J_coord = jacobian_planar_coords(q, L)
        qdot = damped_pinv(J_coord, damping) @ (gain * error)
        qdot = np.clip(qdot, -max_qdot, max_qdot)

        error_hist.append(error.copy())
        error_norm.append(np.linalg.norm(error))
        qdot_hist.append(qdot.copy())
        qdot_norm.append(np.linalg.norm(qdot))

        if converged_step is None and np.linalg.norm(error) < 1e-3:
            converged_step = k + 1

        q = wrap_to_pi(q + dt * qdot)
        q_hist.append(q.copy())
        pose_hist.append(fk_planar_coords(q, L))

    return {
        'q_hist': np.array(q_hist),
        'pose_hist': np.array(pose_hist),
        'error_hist': np.array(error_hist),
        'error_norm': np.array(error_norm),
        'qdot_hist': np.array(qdot_hist),
        'qdot_norm': np.array(qdot_norm),
        'converged_step': converged_step,
    }


def simulate_planar_lie_control(
    q0,
    target_pose,
    L,
    dt=DT_PLANAR,
    gain=PLANAR_GAIN,
    damping=PLANAR_DAMPING,
    steps=PLANAR_STEPS,
    max_qdot=PLANAR_MAX_QDOT,
):
    """Roll out the Lie-theoretic body-twist controller on the planar 4R arm."""
    q = np.asarray(q0, dtype=float).copy()
    T_target = pose_se2_from_xyphi(target_pose)

    q_hist = [q.copy()]
    pose_hist = [fk_planar_coords(q, L)]
    error_hist, error_norm = [], []
    qdot_hist, qdot_norm = [], []
    converged_step = None

    for k in range(steps):
        T_current = pose_se2_from_q(q, L)
        error = lie_pose_error_body(T_current, T_target)
        J_body = jacobian_planar_body(q, L)
        qdot = damped_pinv(J_body, damping) @ (gain * error)
        qdot = np.clip(qdot, -max_qdot, max_qdot)

        error_hist.append(error.copy())
        error_norm.append(np.linalg.norm(error))
        qdot_hist.append(qdot.copy())
        qdot_norm.append(np.linalg.norm(qdot))

        if converged_step is None and np.linalg.norm(error) < 1e-3:
            converged_step = k + 1

        q = wrap_to_pi(q + dt * qdot)
        q_hist.append(q.copy())
        pose_hist.append(fk_planar_coords(q, L))

    return {
        'q_hist': np.array(q_hist),
        'pose_hist': np.array(pose_hist),
        'error_hist': np.array(error_hist),
        'error_norm': np.array(error_norm),
        'qdot_hist': np.array(qdot_hist),
        'qdot_norm': np.array(qdot_norm),
        'converged_step': converged_step,
    }


def draw_planar_arm(ax, q, L, color, label=None, alpha=1.0, linestyle='-'):
    """Draw a planar serial arm on a Matplotlib axes."""
    points = all_joint_positions(q, L)
    ax.plot(
        points[:, 0],
        points[:, 1],
        'o' + linestyle,
        color=color,
        label=label,
        alpha=alpha,
        linewidth=2.5,
        markersize=5,
    )
    ax.plot(0.0, 0.0, 'ks', markersize=7, alpha=alpha)
    ax.plot(points[-1, 0], points[-1, 1], 'o', color=color, alpha=alpha, markersize=8)


def print_planar_summary(name, result):
    """Print compact scalar metrics for a planar rollout."""
    print(
        f"{name:10s} | converged step = {result['converged_step']!s:>4} | "
        f"final ||e|| = {result['error_norm'][-1]:.2e} | "
        f"peak ||qdot|| = {result['qdot_norm'].max():.2f} | "
        f"EE path length = {ee_path_length(result['pose_hist']):.3f}"
    )


---
## 3. Verification

Before comparing controllers, verify the underlying geometry and differential kinematics:

- `fk_planar_coords(q)` matches `pose_se2_from_q(q)` exactly.
- `jacobian_planar_coords(q)` matches centered finite differences of $(x, y, \phi)$.
- `jacobian_planar_space(q)` matches a finite-difference estimate of $\mathrm{vee}(\dot{T}T^{-1})$.
- `jacobian_planar_body(q)` matches a finite-difference estimate of $\mathrm{vee}(T^{-1}\dot{T})$.
- `log_se2(exp_se2(xi)) \approx xi` away from the branch-cut edge case.


In [ ]:
# ========================== Planar Verification ==========================

fk_pose_err = 0.0
coord_jac_err = 0.0
space_jac_err = 0.0
body_jac_err = 0.0
se2_logexp_err = 0.0

for _ in range(1000):
    q = rng.uniform(-np.pi, np.pi, N_JOINTS)
    pose = fk_planar_coords(q, LINK_LENGTHS)
    T = pose_se2_from_q(q, LINK_LENGTHS)
    pose_from_T = pose_coords_from_se2(T)
    fk_pose_err = max(fk_pose_err, np.max(np.abs(pose - pose_from_T)))

q_test = np.array([0.4, -0.6, 0.8, -0.3])
J_coord = jacobian_planar_coords(q_test, LINK_LENGTHS)
J_coord_fd = np.zeros_like(J_coord)
for k in range(N_JOINTS):
    dq = np.zeros(N_JOINTS)
    dq[k] = FD_EPSILON
    pose_plus = fk_planar_coords(q_test + dq, LINK_LENGTHS)
    pose_minus = fk_planar_coords(q_test - dq, LINK_LENGTHS)
    diff = (pose_plus - pose_minus) / (2.0 * FD_EPSILON)
    diff[2] = wrap_to_pi(pose_plus[2] - pose_minus[2]) / (2.0 * FD_EPSILON)
    J_coord_fd[:, k] = diff
coord_jac_err = np.max(np.abs(J_coord - J_coord_fd))

for _ in range(300):
    q = rng.uniform(-np.pi, np.pi, N_JOINTS)
    qdot = rng.standard_normal(N_JOINTS)

    T0 = pose_se2_from_q(q, LINK_LENGTHS)
    T1 = pose_se2_from_q(q + FD_EPSILON * qdot, LINK_LENGTHS)

    V_space_fd = vee_se2(log_se2(T1 @ inv_se2(T0))) / FD_EPSILON
    V_body_fd = vee_se2(log_se2(inv_se2(T0) @ T1)) / FD_EPSILON

    V_space_pred = jacobian_planar_space(q, LINK_LENGTHS) @ qdot
    V_body_pred = jacobian_planar_body(q, LINK_LENGTHS) @ qdot

    space_jac_err = max(space_jac_err, np.max(np.abs(V_space_fd - V_space_pred)))
    body_jac_err = max(body_jac_err, np.max(np.abs(V_body_fd - V_body_pred)))

for _ in range(500):
    xi = np.array([
        rng.uniform(-1.0, 1.0),
        rng.uniform(-1.0, 1.0),
        rng.uniform(-1.2, 1.2),
    ])
    xi_rec = vee_se2(log_se2(exp_se2(xi)))
    se2_logexp_err = max(se2_logexp_err, np.max(np.abs(xi - xi_rec)))

checks = [
    ('Planar FK == SE(2) pose', fk_pose_err, 1e-12),
    ('Coordinate Jacobian FD', coord_jac_err, 1e-6),
    ('Spatial twist Jacobian FD', space_jac_err, 1e-5),
    ('Body twist Jacobian FD', body_jac_err, 1e-5),
    ('log(exp(xi)) on SE(2)', se2_logexp_err, 1e-10),
]

for label, err, tol in checks:
    status = 'PASS' if err < tol else 'FAIL'
    print(f"{label:28s} | max error = {err:.2e} | tol = {tol:.1e} [{status}]")


---
## 4. Easy Planar Tracking Task

First use a moderate target that is well inside the workspace. This is the regime where both controllers should behave well.

If Lie methods were only useful when the coordinate method completely fails, that would be too narrow. A better question is:

- do both controllers converge?
- do they use similar or different transients?
- does the Lie controller obtain a slightly more economical end-effector path or lower peak joint speed?


In [ ]:
# ========================== Easy Planar Target ==========================

result_easy_coord = simulate_planar_coordinate_control(Q0_PLANAR, TARGET_EASY, LINK_LENGTHS)
result_easy_lie = simulate_planar_lie_control(Q0_PLANAR, TARGET_EASY, LINK_LENGTHS)

print_planar_summary('Coordinate', result_easy_coord)
print_planar_summary('Lie', result_easy_lie)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
draw_planar_arm(ax, Q0_PLANAR, LINK_LENGTHS, color='0.7', label='Initial arm', alpha=0.6)
draw_planar_arm(ax, result_easy_coord['q_hist'][-1], LINK_LENGTHS, color='steelblue', label='Coordinate final')
draw_planar_arm(ax, result_easy_lie['q_hist'][-1], LINK_LENGTHS, color='seagreen', label='Lie final', linestyle='--')
ax.plot(result_easy_coord['pose_hist'][:, 0], result_easy_coord['pose_hist'][:, 1], color='steelblue', label='Coordinate EE path')
ax.plot(result_easy_lie['pose_hist'][:, 0], result_easy_lie['pose_hist'][:, 1], color='seagreen', linestyle='--', label='Lie EE path')
ax.plot(TARGET_EASY[0], TARGET_EASY[1], 'r*', markersize=14, label='Target')
ax.set_aspect('equal')
ax.set_title('End-effector path')
ax.legend(fontsize=9)

steps = np.arange(len(result_easy_coord['error_norm']))
axes[1].plot(steps, result_easy_coord['error_norm'], label='Coordinate', color='steelblue')
axes[1].plot(steps, result_easy_lie['error_norm'], label='Lie', color='seagreen', linestyle='--')
axes[1].set_yscale('log')
axes[1].set_xlabel('Step')
axes[1].set_ylabel(r'$\|e\|$')
axes[1].set_title('Task error norm')
axes[1].legend(fontsize=9)

axes[2].plot(steps, result_easy_coord['qdot_norm'], label='Coordinate', color='steelblue')
axes[2].plot(steps, result_easy_lie['qdot_norm'], label='Lie', color='seagreen', linestyle='--')
axes[2].set_xlabel('Step')
axes[2].set_ylabel(r'$\|\dot{q}\|$')
axes[2].set_title('Joint-speed norm')
axes[2].legend(fontsize=9)

plt.suptitle('Easy planar target: both controllers converge')
plt.tight_layout()
plt.show()


---
## 5. Large Orientation Change in the Plane

Now use a target that demands a much larger pose change. In the plane, the wrapped-angle baseline is still viable, so the difference is not about total failure.

Instead, the comparison is about the **transient geometry**:

- coordinate control uses a chart error in $(x, y, \phi)$,
- Lie control uses a body-frame logarithmic error in $SE(2)$.

This often changes the tradeoff between convergence speed, end-effector path length, and peak joint velocity.


In [ ]:
# ========================== Large Orientation Change ==========================

result_large_coord = simulate_planar_coordinate_control(Q0_PLANAR, TARGET_LARGE, LINK_LENGTHS)
result_large_lie = simulate_planar_lie_control(Q0_PLANAR, TARGET_LARGE, LINK_LENGTHS)

print_planar_summary('Coordinate', result_large_coord)
print_planar_summary('Lie', result_large_lie)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
draw_planar_arm(ax, Q0_PLANAR, LINK_LENGTHS, color='0.7', label='Initial arm', alpha=0.6)
draw_planar_arm(ax, result_large_coord['q_hist'][-1], LINK_LENGTHS, color='steelblue', label='Coordinate final')
draw_planar_arm(ax, result_large_lie['q_hist'][-1], LINK_LENGTHS, color='seagreen', label='Lie final', linestyle='--')
ax.plot(result_large_coord['pose_hist'][:, 0], result_large_coord['pose_hist'][:, 1], color='steelblue', label='Coordinate EE path')
ax.plot(result_large_lie['pose_hist'][:, 0], result_large_lie['pose_hist'][:, 1], color='seagreen', linestyle='--', label='Lie EE path')
ax.plot(TARGET_LARGE[0], TARGET_LARGE[1], 'r*', markersize=14, label='Target')
ax.set_aspect('equal')
ax.set_title('End-effector path')
ax.legend(fontsize=9)

steps = np.arange(len(result_large_coord['error_norm']))
axes[1].plot(steps, result_large_coord['error_norm'], label='Coordinate', color='steelblue')
axes[1].plot(steps, result_large_lie['error_norm'], label='Lie', color='seagreen', linestyle='--')
axes[1].set_yscale('log')
axes[1].set_xlabel('Step')
axes[1].set_ylabel(r'$\|e\|$')
axes[1].set_title('Task error norm')
axes[1].legend(fontsize=9)

axes[2].plot(steps, result_large_coord['qdot_norm'], label='Coordinate', color='steelblue')
axes[2].plot(steps, result_large_lie['qdot_norm'], label='Lie', color='seagreen', linestyle='--')
axes[2].set_xlabel('Step')
axes[2].set_ylabel(r'$\|\dot{q}\|$')
axes[2].set_title('Joint-speed norm')
axes[2].legend(fontsize=9)

plt.suptitle('Large planar pose change: same robot, same gains, different error geometry')
plt.tight_layout()
plt.show()


---
## 6. Rotating the Entire World Frame

A subtle advantage of Lie errors is **representation consistency**.

Here we rotate the entire world by a fixed angle about the base. The physical task is unchanged; only the external chart changed.

What should happen?

- The raw coordinate error vector changes because its translational components are tied to the world axes.
- The body-frame Lie error stays essentially unchanged, because it is defined relative to the current pose itself.
- Both controllers still produce the same control action when their Jacobians are transformed consistently.

This is a good example of Lie methods improving the *meaning* of the error, even in a case where a well-implemented coordinate controller can still behave correctly.


In [ ]:
# ========================== Frame Rotation Consistency ==========================

T_world_rot = pose_se2_from_xyphi(np.array([0.0, 0.0, FRAME_ROTATION]))

pose_ref = fk_planar_coords(Q_FRAME, LINK_LENGTHS)
T_ref = pose_se2_from_q(Q_FRAME, LINK_LENGTHS)
T_target = pose_se2_from_xyphi(TARGET_FRAME)

# Unrotated-frame errors and control actions
e_coord = coordinate_pose_error(pose_ref, TARGET_FRAME)
J_coord = jacobian_planar_coords(Q_FRAME, LINK_LENGTHS)
qdot_coord = damped_pinv(J_coord, PLANAR_DAMPING) @ (PLANAR_GAIN * e_coord)

e_lie = lie_pose_error_body(T_ref, T_target)
J_body = jacobian_planar_body(Q_FRAME, LINK_LENGTHS)
qdot_lie = damped_pinv(J_body, PLANAR_DAMPING) @ (PLANAR_GAIN * e_lie)

# Rotated world-frame representation of the same physical task
T_ref_rot = T_world_rot @ T_ref
T_target_rot = T_world_rot @ T_target
pose_ref_rot = pose_coords_from_se2(T_ref_rot)
target_rot = pose_coords_from_se2(T_target_rot)

e_coord_rot = coordinate_pose_error(pose_ref_rot, target_rot)
R_world = rot2(FRAME_ROTATION)
S_coord = np.eye(3)
S_coord[:2, :2] = R_world
J_coord_rot = S_coord @ J_coord
qdot_coord_rot = damped_pinv(J_coord_rot, PLANAR_DAMPING) @ (PLANAR_GAIN * e_coord_rot)

e_lie_rot = lie_pose_error_body(T_ref_rot, T_target_rot)
J_space_rot = adjoint_se2(T_world_rot) @ jacobian_planar_space(Q_FRAME, LINK_LENGTHS)
J_body_rot = adjoint_se2(inv_se2(T_ref_rot)) @ J_space_rot
qdot_lie_rot = damped_pinv(J_body_rot, PLANAR_DAMPING) @ (PLANAR_GAIN * e_lie_rot)

metrics = {
    r'$\|e_{coord} - e_{coord}^{rot}\|$': np.linalg.norm(e_coord - e_coord_rot),
    r'$\|e_{Lie} - e_{Lie}^{rot}\|$': np.linalg.norm(e_lie - e_lie_rot),
    r'$\|\dot{q}_{coord} - \dot{q}_{coord}^{rot}\|$': np.linalg.norm(qdot_coord - qdot_coord_rot),
    r'$\|\dot{q}_{Lie} - \dot{q}_{Lie}^{rot}\|$': np.linalg.norm(qdot_lie - qdot_lie_rot),
}

for label, value in metrics.items():
    print(f"{label:42s} = {value:.2e}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].bar(['Coordinate', 'Lie'], [np.linalg.norm(e_coord - e_coord_rot), np.linalg.norm(e_lie - e_lie_rot)], color=['steelblue', 'seagreen'])
axes[0].set_yscale('log')
axes[0].set_ylabel('Difference under world rotation')
axes[0].set_title('Error representation changes')

axes[1].bar(['Coordinate', 'Lie'], [np.linalg.norm(qdot_coord - qdot_coord_rot), np.linalg.norm(qdot_lie - qdot_lie_rot)], color=['steelblue', 'seagreen'])
axes[1].set_yscale('log')
axes[1].set_ylabel('Difference under world rotation')
axes[1].set_title('Control action stays invariant')

plt.suptitle('Rotating the chart vs changing the physics')
plt.tight_layout()
plt.show()


---
## 7. Moving to 3D: Where Euler Coordinates Become Fragile

In the planar case, the orientation state is only one angle on $S^1$, so wrapped subtraction already fixes the main branch-cut issue.

The real geometric advantage of Lie methods becomes much clearer in 3D, where orientation charts such as ZYX Euler angles are singular at

$$\theta = \pm \frac{\pi}{2}.$$

We now compare:

- a **coordinate orientation controller** that defines error in ZYX Euler angles and converts Euler-rate commands to angular velocity using the Euler-rate matrix,
- a **Lie controller** that defines error with
  $$e_R = \mathrm{vee}(\log(R^\top R_d))$$
  and updates orientation via Rodrigues' exponential.


In [ ]:
# ========================== SO(3) Helpers ==========================

def rot_x(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([
        [1.0, 0.0, 0.0],
        [0.0, c, -s],
        [0.0, s, c],
    ])


def rot_y(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([
        [c, 0.0, s],
        [0.0, 1.0, 0.0],
        [-s, 0.0, c],
    ])


def rot_z(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([
        [c, -s, 0.0],
        [s, c, 0.0],
        [0.0, 0.0, 1.0],
    ])


def hat_so3(omega):
    wx, wy, wz = omega
    return np.array([
        [0.0, -wz, wy],
        [wz, 0.0, -wx],
        [-wy, wx, 0.0],
    ])


def vee_so3(W):
    return np.array([W[2, 1], W[0, 2], W[1, 0]])


def exp_so3(omega):
    """Rodrigues formula for SO(3)."""
    theta = np.linalg.norm(omega)
    W = hat_so3(omega)
    if theta < 1e-10:
        return np.eye(3) + W + 0.5 * (W @ W)
    A = np.sin(theta) / theta
    B = (1.0 - np.cos(theta)) / (theta ** 2)
    return np.eye(3) + A * W + B * (W @ W)


def log_so3(R):
    """Closed-form matrix logarithm on SO(3)."""
    cos_theta = np.clip((np.trace(R) - 1.0) / 2.0, -1.0, 1.0)
    theta = np.arccos(cos_theta)
    if theta < 1e-10:
        return np.zeros((3, 3))
    if np.pi - theta < 1e-6:
        diag = np.diagonal(R)
        axis = np.sqrt(np.maximum((diag + 1.0) / 2.0, 0.0))
        if axis[0] > 1e-6:
            axis[1] = np.copysign(axis[1], R[0, 1] + R[1, 0])
            axis[2] = np.copysign(axis[2], R[0, 2] + R[2, 0])
        elif axis[1] > 1e-6:
            axis[0] = np.copysign(axis[0], R[0, 1] + R[1, 0])
            axis[2] = np.copysign(axis[2], R[1, 2] + R[2, 1])
        else:
            axis[0] = np.copysign(axis[0], R[0, 2] + R[2, 0])
            axis[1] = np.copysign(axis[1], R[1, 2] + R[2, 1])
        axis = axis / np.linalg.norm(axis)
        return hat_so3(theta * axis)
    return (theta / (2.0 * np.sin(theta))) * (R - R.T)


def euler_zyx_to_R(euler):
    """R = Rz(yaw) Ry(pitch) Rx(roll)."""
    roll, pitch, yaw = euler
    return rot_z(yaw) @ rot_y(pitch) @ rot_x(roll)


def R_to_euler_zyx(R):
    """Principal-value ZYX Euler angles in the conventional pitch range [-pi/2, pi/2]."""
    pitch = np.arctan2(-R[2, 0], np.sqrt(R[0, 0]**2 + R[1, 0]**2))
    cp = np.cos(pitch)
    if abs(cp) < 1e-8:
        roll = 0.0
        yaw = np.arctan2(-R[0, 1], R[1, 1])
    else:
        roll = np.arctan2(R[2, 1], R[2, 2])
        yaw = np.arctan2(R[1, 0], R[0, 0])
    return np.array([wrap_to_pi(roll), wrap_to_pi(pitch), wrap_to_pi(yaw)])


def euler_rate_matrix_zyx(euler):
    """Map Euler-angle rates [roll_dot, pitch_dot, yaw_dot] to body angular velocity."""
    roll, pitch, _ = euler
    sr, cr = np.sin(roll), np.cos(roll)
    sp, cp = np.sin(pitch), np.cos(pitch)
    return np.array([
        [1.0, 0.0, -sp],
        [0.0, cr, sr * cp],
        [0.0, -sr, cr * cp],
    ])


def clip_vector_norm(v, max_norm):
    """Clip a vector by Euclidean norm."""
    norm = np.linalg.norm(v)
    if norm <= max_norm:
        return v
    return v * (max_norm / norm)


def rotation_distance(R_a, R_b):
    """Geodesic distance induced by the SO(3) logarithm."""
    return np.linalg.norm(vee_so3(log_so3(R_a.T @ R_b)))


def simulate_so3_coordinate_control(
    euler0,
    euler_target,
    dt=DT_SO3,
    gain=SO3_GAIN,
    steps=SO3_STEPS,
    max_omega=SO3_MAX_OMEGA,
):
    """Closed-loop orientation control using Euler-angle error and the Euler-rate matrix."""
    R = euler_zyx_to_R(euler0)
    R_target = euler_zyx_to_R(euler_target)

    euler_hist = []
    coord_error_hist = []
    geo_error_hist = []
    cond_hist = []
    omega_cmd_norm = []
    omega_norm = []

    for _ in range(steps):
        euler = R_to_euler_zyx(R)
        coord_error = wrap_to_pi(euler_target - euler)
        E = euler_rate_matrix_zyx(euler)
        omega_cmd = E @ (gain * coord_error)
        omega = clip_vector_norm(omega_cmd, max_omega)

        euler_hist.append(euler.copy())
        coord_error_hist.append(coord_error.copy())
        geo_error_hist.append(rotation_distance(R, R_target))
        cond_hist.append(np.linalg.cond(E))
        omega_cmd_norm.append(np.linalg.norm(omega_cmd))
        omega_norm.append(np.linalg.norm(omega))

        R = R @ exp_so3(omega * dt)

    return {
        'euler_hist': np.array(euler_hist),
        'coord_error_hist': np.array(coord_error_hist),
        'geo_error_hist': np.array(geo_error_hist),
        'cond_hist': np.array(cond_hist),
        'omega_cmd_norm': np.array(omega_cmd_norm),
        'omega_norm': np.array(omega_norm),
    }


def simulate_so3_lie_control(
    euler0,
    euler_target,
    dt=DT_SO3,
    gain=SO3_GAIN,
    steps=SO3_STEPS,
    max_omega=SO3_MAX_OMEGA,
):
    """Closed-loop orientation control using Lie-algebra error on SO(3)."""
    R = euler_zyx_to_R(euler0)
    R_target = euler_zyx_to_R(euler_target)

    lie_error_hist = []
    geo_error_hist = []
    omega_cmd_norm = []
    omega_norm = []

    for _ in range(steps):
        lie_error = vee_so3(log_so3(R.T @ R_target))
        omega_cmd = gain * lie_error
        omega = clip_vector_norm(omega_cmd, max_omega)

        lie_error_hist.append(lie_error.copy())
        geo_error_hist.append(rotation_distance(R, R_target))
        omega_cmd_norm.append(np.linalg.norm(omega_cmd))
        omega_norm.append(np.linalg.norm(omega))

        R = R @ exp_so3(omega * dt)

    return {
        'lie_error_hist': np.array(lie_error_hist),
        'geo_error_hist': np.array(geo_error_hist),
        'omega_cmd_norm': np.array(omega_cmd_norm),
        'omega_norm': np.array(omega_norm),
    }


---
## 8. SO(3) Verification and Static Sweep Through the Euler Singularity

First verify that the closed-form $SO(3)$ logarithm/exponential pair is working.

Then sweep a *physical* pitch motion from $60^\circ$ to $120^\circ$ using the rotation matrices $R_y(\theta)$.

This is the important point: the physical motion is perfectly smooth, but the principal-value ZYX Euler representation must jump as the path crosses the chart boundary near $90^\circ$.


In [ ]:
# ========================== SO(3) Verification + Singular Sweep ==========================

so3_logexp_err = 0.0
for _ in range(500):
    axis = rng.standard_normal(3)
    axis /= np.linalg.norm(axis)
    theta = rng.uniform(0.0, 2.6)
    R = exp_so3(theta * axis)
    R_rec = exp_so3(vee_so3(log_so3(R)))
    so3_logexp_err = max(so3_logexp_err, np.max(np.abs(R - R_rec)))

status = 'PASS' if so3_logexp_err < 1e-10 else 'FAIL'
print(f"SO(3) exp(log(R))             | max error = {so3_logexp_err:.2e} | tol = 1.0e-10 [{status}]")

pitch_path_deg = np.linspace(60.0, 120.0, 241)
pitch_path = np.deg2rad(pitch_path_deg)
R_path = [rot_y(theta) for theta in pitch_path]

euler_repr = np.array([R_to_euler_zyx(R) for R in R_path])
cond_path = np.array([np.linalg.cond(euler_rate_matrix_zyx(e)) for e in euler_repr])
coord_error_path = np.array([wrap_to_pi(-e) for e in euler_repr])
lie_error_path = np.array([vee_so3(log_so3(R)) for R in R_path])

roll_jump = np.max(np.abs(np.diff(euler_repr[:, 0])))
yaw_jump = np.max(np.abs(np.diff(euler_repr[:, 2])))
print(f"Max roll jump across sweep    = {roll_jump:.3f} rad")
print(f"Max yaw jump across sweep     = {yaw_jump:.3f} rad")
print(f"Max cond(Euler-rate matrix)   = {np.max(cond_path):.2e}")

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].plot(pitch_path_deg, np.rad2deg(euler_repr[:, 0]), label='Recovered roll')
axes[0, 0].plot(pitch_path_deg, np.rad2deg(euler_repr[:, 1]), label='Recovered pitch')
axes[0, 0].plot(pitch_path_deg, np.rad2deg(euler_repr[:, 2]), label='Recovered yaw')
axes[0, 0].axvline(90.0, color='k', linestyle=':', alpha=0.6)
axes[0, 0].set_xlabel('Physical pitch path (deg)')
axes[0, 0].set_ylabel('Principal Euler angles (deg)')
axes[0, 0].set_title('Euler chart jumps at the boundary')
axes[0, 0].legend(fontsize=9)

axes[0, 1].semilogy(pitch_path_deg, cond_path, color='darkorange')
axes[0, 1].axvline(90.0, color='k', linestyle=':', alpha=0.6)
axes[0, 1].set_xlabel('Physical pitch path (deg)')
axes[0, 1].set_ylabel(r'$\kappa(E)$')
axes[0, 1].set_title('Euler-rate conditioning blows up')

axes[1, 0].plot(pitch_path_deg, coord_error_path[:, 0], label='roll error')
axes[1, 0].plot(pitch_path_deg, coord_error_path[:, 1], label='pitch error')
axes[1, 0].plot(pitch_path_deg, coord_error_path[:, 2], label='yaw error')
axes[1, 0].axvline(90.0, color='k', linestyle=':', alpha=0.6)
axes[1, 0].set_xlabel('Physical pitch path (deg)')
axes[1, 0].set_ylabel('Coordinate error to identity (rad)')
axes[1, 0].set_title('Euler error components are discontinuous')
axes[1, 0].legend(fontsize=9)

axes[1, 1].plot(pitch_path_deg, lie_error_path[:, 0], label='x component')
axes[1, 1].plot(pitch_path_deg, lie_error_path[:, 1], label='y component')
axes[1, 1].plot(pitch_path_deg, lie_error_path[:, 2], label='z component')
axes[1, 1].axvline(90.0, color='k', linestyle=':', alpha=0.6)
axes[1, 1].set_xlabel('Physical pitch path (deg)')
axes[1, 1].set_ylabel('Lie error to identity (rad)')
axes[1, 1].set_title('Lie error stays smooth on the manifold')
axes[1, 1].legend(fontsize=9)

plt.suptitle('Static sweep: smooth physical motion, discontinuous Euler chart')
plt.tight_layout()
plt.show()


---
## 9. Closed-Loop SO(3) Tracking Near Gimbal Lock

Finally, run both orientation controllers from an initial orientation very close to the ZYX singularity.

The Lie controller still works with the same logarithmic error it used everywhere else.

The coordinate controller can still converge here, but it inherits two visible issues from the Euler chart:

- the Euler-rate matrix becomes badly conditioned,
- the commanded angular velocity can become much more sensitive to small coordinate changes.


In [ ]:
# ========================== Closed-Loop SO(3) Comparison ==========================

result_so3_coord = simulate_so3_coordinate_control(EULER0_SO3, EULERD_SO3)
result_so3_lie = simulate_so3_lie_control(EULER0_SO3, EULERD_SO3)

print(f"Coordinate controller | initial geodesic error = {result_so3_coord['geo_error_hist'][0]:.3f} | "
      f"step 20 = {result_so3_coord['geo_error_hist'][19]:.3f} | final = {result_so3_coord['geo_error_hist'][-1]:.3e}")
print(f"Lie controller        | initial geodesic error = {result_so3_lie['geo_error_hist'][0]:.3f} | "
      f"step 20 = {result_so3_lie['geo_error_hist'][19]:.3f} | final = {result_so3_lie['geo_error_hist'][-1]:.3e}")
print(f"Max cond(E) along coordinate rollout = {np.max(result_so3_coord['cond_hist']):.2e}")
print(f"Max ||omega_cmd|| coordinate / lie   = {np.max(result_so3_coord['omega_cmd_norm']):.2f} / {np.max(result_so3_lie['omega_cmd_norm']):.2f}")

steps = np.arange(SO3_STEPS)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(steps, result_so3_coord['geo_error_hist'], label='Coordinate', color='steelblue')
axes[0].plot(steps, result_so3_lie['geo_error_hist'], label='Lie', color='seagreen', linestyle='--')
axes[0].set_yscale('log')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Geodesic error')
axes[0].set_title('Orientation tracking error')
axes[0].legend(fontsize=9)

axes[1].plot(steps, result_so3_coord['omega_cmd_norm'], label='Coordinate raw command', color='steelblue')
axes[1].plot(steps, result_so3_lie['omega_cmd_norm'], label='Lie raw command', color='seagreen', linestyle='--')
axes[1].set_xlabel('Step')
axes[1].set_ylabel(r'$\|\omega_{cmd}\|$')
axes[1].set_title('Angular-velocity demand')
axes[1].legend(fontsize=9)

axes[2].semilogy(steps, result_so3_coord['cond_hist'], color='darkorange')
axes[2].set_xlabel('Step')
axes[2].set_ylabel(r'$\kappa(E)$')
axes[2].set_title('Euler-rate conditioning along the rollout')

plt.suptitle('Near-gimbal-lock SO(3) tracking')
plt.tight_layout()
plt.show()


---
## 10. Takeaways

The comparison shows three different regimes:

1. **Planar, moderate targets:** both controllers work well. This is why coordinate methods remain practical for simple tasks.
2. **Planar, larger pose changes:** the Lie controller changes the transient geometry. It often reduces end-effector path length or peak joint speed, but the main advantage is conceptual consistency, not magical performance.
3. **3D orientation near singularities:** Euler coordinates become the real bottleneck. Their chart jumps and poor conditioning are representation problems, not robot-mechanics problems. The Lie controller avoids those by defining error directly on $SO(3)$.

The most important lesson is therefore:

> Lie methods are not valuable because they always make every controller faster. They are valuable because they define motion and error on the **correct geometry** of rigid-body pose, which becomes increasingly important as the task moves from planar intuition to full 3D orientation and pose control.
